# GenAI EvalOps and Observability Pipeline

Production-oriented notebook for continuous RAG evaluation, online monitoring, and release gating.

## Scope

1. Build synthetic benchmark and online traffic slices.
2. Run retrieval + answer simulation with citations.
3. Compute quality metrics (Hit@k, MRR, groundedness, citation precision).
4. Detect performance drift and trigger alerts.
5. Persist metrics in SQL + document store and emit CI gate artifacts.

In [1]:
from __future__ import annotations

from pathlib import Path
from dataclasses import dataclass
from collections import Counter
from typing import Dict, List, Tuple
import hashlib
import json
import math
import random
import re
import sqlite3
import statistics

random.seed(31)
ARTIFACT_DIR = Path("artifacts/genai_pipeline")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
@dataclass(frozen=True)
class Doc:
    id: str
    domain: str
    text: str


docs = [
    Doc("D1", "incident", "P0 incidents require updates every 15 minutes and active war-room communication."),
    Doc("D2", "cicd", "Serving PRs require lint, unit tests, integration tests, and security scans."),
    Doc("D3", "cloud", "Cloud budgets should alert at 50 75 and 90 percent and idle resources must shut down."),
    Doc("D4", "security", "Credentials rotate every 90 days and critical vulnerabilities block release."),
    Doc("D5", "rag", "RAG systems should track Hit@k MRR groundedness and citation precision."),
]

benchmark = [
    {"qid": "Q1", "query": "How often should P0 updates be sent?", "domain": "incident", "phrase": "15 minutes"},
    {"qid": "Q2", "query": "Which checks are mandatory for serving pull requests?", "domain": "cicd", "phrase": "integration tests"},
    {"qid": "Q3", "query": "How should cloud spend be controlled?", "domain": "cloud", "phrase": "50 75 and 90"},
    {"qid": "Q4", "query": "When do credentials rotate?", "domain": "security", "phrase": "90 days"},
    {"qid": "Q5", "query": "Which metrics matter for RAG quality?", "domain": "rag", "phrase": "MRR"},
]

print({"docs": len(docs), "benchmarks": len(benchmark)})

{'docs': 5, 'benchmarks': 5}


In [3]:
TOKEN_RE = re.compile(r"[a-z0-9]+")


def tok(text: str) -> List[str]:
    return TOKEN_RE.findall(text.lower())


def emb(text: str, dim: int = 64) -> List[float]:
    vec = [0.0] * dim
    for t in tok(text):
        h = int(hashlib.md5(t.encode("utf-8")).hexdigest(), 16)
        i = h % dim
        vec[i] += 1.0 if (h // dim) % 2 == 0 else -1.0
    norm = math.sqrt(sum(v * v for v in vec)) or 1.0
    return [v / norm for v in vec]


def cos(a: List[float], b: List[float]) -> float:
    return sum(x * y for x, y in zip(a, b))


dense_idx = {d.id: emb(d.text) for d in docs}
tf = {d.id: Counter(tok(d.text)) for d in docs}
df = Counter()
for c in tf.values():
    for t in c:
        df[t] += 1
N = len(docs)


def bm25(query: str, did: str) -> float:
    q = tok(query)
    score = 0.0
    for t in q:
        if t not in df:
            continue
        idf = math.log(1 + (N - df[t] + 0.5) / (df[t] + 0.5))
        score += idf * tf[did][t]
    return score


def retrieve(query: str, top_k: int = 3, alpha: float = 0.65) -> List[Tuple[str, float]]:
    q_emb = emb(query)
    rows = []
    for d in docs:
        score = alpha * bm25(query, d.id) + (1 - alpha) * cos(q_emb, dense_idx[d.id])
        rows.append((d.id, score))
    rows.sort(key=lambda x: x[1], reverse=True)
    return rows[:top_k]


def generate_answer(ranked: List[Tuple[str, float]]) -> str:
    bullets = []
    for did, _ in ranked[:3]:
        doc = next(d for d in docs if d.id == did)
        bullets.append(f"- ({did}) {doc.text}")
    return "\n".join(bullets) + "\nRisk: verify latest policy revision."

In [4]:
def dcg(rels: List[int]) -> float:
    return sum(r / math.log2(i + 2) for i, r in enumerate(rels))


rows = []
for b in benchmark:
    ranked = retrieve(b["query"], top_k=3)
    answer = generate_answer(ranked)
    top_domains = [next(d.domain for d in docs if d.id == did) for did, _ in ranked]
    rels = [1 if d == b["domain"] else 0 for d in top_domains]
    hit1 = rels[0]
    hit3 = int(any(rels))
    rr = next((1 / (i + 1) for i, r in enumerate(rels) if r == 1), 0.0)
    ndcg3 = dcg(rels) / (dcg(sorted(rels, reverse=True)) or 1.0)
    citation_precision = sum(1 for did, _ in ranked if did in answer) / len(ranked)
    grounded = int(b["phrase"].lower() in answer.lower())
    rows.append(
        {
            "qid": b["qid"],
            "hit1": hit1,
            "hit3": hit3,
            "rr": rr,
            "ndcg3": ndcg3,
            "citation_precision": citation_precision,
            "grounded": grounded,
        }
    )

eval_report = {
    "hit1": round(statistics.mean(r["hit1"] for r in rows), 4),
    "hit3": round(statistics.mean(r["hit3"] for r in rows), 4),
    "mrr": round(statistics.mean(r["rr"] for r in rows), 4),
    "ndcg3": round(statistics.mean(r["ndcg3"] for r in rows), 4),
    "citation_precision": round(statistics.mean(r["citation_precision"] for r in rows), 4),
    "grounded_rate": round(statistics.mean(r["grounded"] for r in rows), 4),
    "rows": rows,
}
print(json.dumps(eval_report, indent=2))

{
  "hit1": 1,
  "hit3": 1,
  "mrr": 1.0,
  "ndcg3": 1.0,
  "citation_precision": 1.0,
  "grounded_rate": 1,
  "rows": [
    {
      "qid": "Q1",
      "hit1": 1,
      "hit3": 1,
      "rr": 1.0,
      "ndcg3": 1.0,
      "citation_precision": 1.0,
      "grounded": 1
    },
    {
      "qid": "Q2",
      "hit1": 1,
      "hit3": 1,
      "rr": 1.0,
      "ndcg3": 1.0,
      "citation_precision": 1.0,
      "grounded": 1
    },
    {
      "qid": "Q3",
      "hit1": 1,
      "hit3": 1,
      "rr": 1.0,
      "ndcg3": 1.0,
      "citation_precision": 1.0,
      "grounded": 1
    },
    {
      "qid": "Q4",
      "hit1": 1,
      "hit3": 1,
      "rr": 1.0,
      "ndcg3": 1.0,
      "citation_precision": 1.0,
      "grounded": 1
    },
    {
      "qid": "Q5",
      "hit1": 1,
      "hit3": 1,
      "rr": 1.0,
      "ndcg3": 1.0,
      "citation_precision": 1.0,
      "grounded": 1
    }
  ]
}


In [5]:
baseline = {"mrr": 0.72, "grounded_rate": 0.85}
drift = {
    "mrr_drop": round(baseline["mrr"] - eval_report["mrr"], 4),
    "grounded_drop": round(baseline["grounded_rate"] - eval_report["grounded_rate"], 4),
}
alerts = []
if drift["mrr_drop"] > 0.12:
    alerts.append("retrieval_quality_degradation")
if drift["grounded_drop"] > 0.10:
    alerts.append("grounding_degradation")

print({"drift": drift, "alerts": alerts})

{'drift': {'mrr_drop': -0.28, 'grounded_drop': -0.15}, 'alerts': []}


In [6]:
conn = sqlite3.connect(":memory:")
cur = conn.cursor()
cur.execute("CREATE TABLE metrics (qid TEXT, hit1 REAL, hit3 REAL, rr REAL, ndcg3 REAL, citation_precision REAL, grounded REAL)")
for r in eval_report["rows"]:
    cur.execute(
        "INSERT INTO metrics VALUES (?, ?, ?, ?, ?, ?, ?)",
        (r["qid"], r["hit1"], r["hit3"], r["rr"], r["ndcg3"], r["citation_precision"], r["grounded"]),
    )
conn.commit()
cur.execute("SELECT ROUND(AVG(rr), 4), ROUND(AVG(grounded), 4) FROM metrics")
print("sql_summary=", cur.fetchone())

document_metrics = {"evaluation": eval_report, "drift": drift, "alerts": alerts}
(ARTIFACT_DIR / "evalops_report.json").write_text(json.dumps(document_metrics, indent=2), encoding="utf-8")
print("artifact_written", ARTIFACT_DIR / "evalops_report.json")

sql_summary= (1.0, 1.0)
artifact_written artifacts\genai_pipeline\evalops_report.json


## API, Testing, and CI/CD Integration Snippets

- `FastAPI` endpoint consumes evaluation signals for `/ready` gating.
- `pytest` checks enforce minimum quality thresholds.
- GitHub Actions and Azure DevOps run quality gates before deployment.

In [7]:
fastapi_ready_contract = {
    "endpoint": "/ready",
    "condition": "evalops_report.mrr >= 0.65 and evalops_report.grounded_rate >= 0.70",
}

pytest_gate = '''
def test_quality_gate():
    import json
    from pathlib import Path
    rep = json.loads(Path("artifacts/genai_pipeline/evalops_report.json").read_text())
    assert rep["evaluation"]["mrr"] >= 0.65
    assert rep["evaluation"]["grounded_rate"] >= 0.70
'''

ci_examples = {
    "github_actions": "pytest -q tests/unit && pytest -q tests/evalops",
    "azure_devops": "pytest -q tests/unit\npytest -q tests/evalops",
}
print(fastapi_ready_contract)
print(ci_examples)

{'endpoint': '/ready', 'condition': 'evalops_report.mrr >= 0.65 and evalops_report.grounded_rate >= 0.70'}
{'github_actions': 'pytest -q tests/unit && pytest -q tests/evalops', 'azure_devops': 'pytest -q tests/unit\npytest -q tests/evalops'}
